# Phase 8 — Deployment Readiness
**Goal:** Package the agent for local deployment, add structured logging/tracing, capture latency, and demonstrate graceful failure handling.

---

## 8.1 Deployment Architecture

```
HTTP Client
    │  POST /chat  {"session_id": ..., "message": ...}
    ▼
FastAPI App (main.py)
    │
    ├── Guardrails check (agent/guardrails.py)
    ├── Session memory lookup (agent/memory.py)
    ├── AgentExecutor.invoke()  (agent/agent.py)
    │       ├── search_knowledge_base → ChromaDB
    │       ├── check_ticket_status   → in-memory store
    │       ├── create_support_ticket → in-memory store
    │       └── escalate_to_human     → in-memory store
    ├── Structured log (logs/agent_interactions.jsonl)
    └── HTTP Response  {"response": ..., "session_id": ...}
```

## 8.2 FastAPI Application
The cell below writes `main.py` to the project root.

In [1]:
MAIN_PY = '''
"""
main.py — TaskFlow Pro Support Agent API
Run: uvicorn main:app --reload --port 8000
"""
import os
import time
import uuid
import logging
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

from agent.agent import build_agent, run_agent_turn
from agent.memory import AgentMemory

# ── Logging setup ────────────────────────────────────────────────
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logger = logging.getLogger("taskflow_agent")

# ── State shared across requests ──────────────────────────────────
executor = None
sessions: dict = {}   # session_id → AgentMemory


@asynccontextmanager
async def lifespan(app: FastAPI):
    global executor
    logger.info("Building agent...")
    executor = build_agent(verbose=False)
    logger.info("Agent ready.")
    yield
    logger.info("Shutting down.")


app = FastAPI(title="TaskFlow Pro Support Agent", version="1.0.0", lifespan=lifespan)


class ChatRequest(BaseModel):
    message: str
    session_id: str = ""


class ChatResponse(BaseModel):
    response: str
    session_id: str
    latency_ms: float


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    if not req.message.strip():
        raise HTTPException(status_code=400, detail="Message cannot be empty.")

    session_id = req.session_id or str(uuid.uuid4())
    if session_id not in sessions:
        sessions[session_id] = AgentMemory()

    memory = sessions[session_id]
    start = time.perf_counter()

    try:
        response = run_agent_turn(executor, memory, req.message)
    except Exception as exc:  # graceful failure
        logger.error("Agent error for session %s: %s", session_id, exc)
        response = (
            "I encountered an unexpected error. Your query has been logged. "
            "Please try again or contact support@taskflowpro.com."
        )

    latency_ms = (time.perf_counter() - start) * 1000
    logger.info("session=%s latency=%.1fms", session_id, latency_ms)
    return ChatResponse(response=response, session_id=session_id, latency_ms=round(latency_ms, 1))


@app.delete("/session/{session_id}")
async def end_session(session_id: str):
    if session_id in sessions:
        sessions[session_id].reset()
        del sessions[session_id]
    return {"status": "session cleared"}


@app.get("/health")
async def health():
    return {"status": "ok", "active_sessions": len(sessions)}
'''

with open("../main.py", "w", encoding="utf-8") as f:
    f.write(MAIN_PY.strip())

print("main.py written.")

main.py written.


## 8.3 Latency Capture
Measure response latency for a sample of queries without running the full API server.

In [3]:
import time
import sys
import os
sys.path.insert(0, os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv("../.env")

from agent.agent import build_agent, run_agent_turn
from agent.memory import AgentMemory
import pandas as pd

executor = build_agent(verbose=False)

latency_queries = [
    "How do I add a sub-task?",
    "What is the refund policy for annual plans?",
    "My Gantt chart is loading slowly.",
    "How do I connect TaskFlow to Microsoft Teams?",
    "I keep getting ERR-403 on every project. Please open a ticket.",
]

records = []
for q in latency_queries:
    mem = AgentMemory()
    t0 = time.perf_counter()
    resp = run_agent_turn(executor, mem, q)
    elapsed = (time.perf_counter() - t0) * 1000
    records.append({"query": q[:60], "latency_ms": round(elapsed, 1), "response_chars": len(resp)})

df = pd.DataFrame(records)
print(df.to_string(index=False))
print(f"\nP50 latency: {df['latency_ms'].median():.0f} ms")
print(f"P95 latency: {df['latency_ms'].quantile(0.95):.0f} ms")
print(f"Max latency: {df['latency_ms'].max():.0f} ms")

Loading existing vector store from 'c:\Users\admin\AgenticAI_Course\Capstone project\data/vectorstore'...


c:\Users\admin\AgenticAI_Course\Capstone project\agent\retriever.py:39: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(persist_directory=VECTOR_STORE_PATH, embedding_function=embeddings)


                                                       query  latency_ms  response_chars
                                    How do I add a sub-task?      8115.3             495
                 What is the refund policy for annual plans?      2838.0             332
                           My Gantt chart is loading slowly.      5728.6             991
               How do I connect TaskFlow to Microsoft Teams?      2538.7             388
I keep getting ERR-403 on every project. Please open a ticke      2798.7             350

P50 latency: 2838 ms
P95 latency: 7638 ms
Max latency: 8115 ms


## 8.4 Graceful Failure Handling

In [4]:
# Simulate a failure scenario: LLM call raises an exception
from unittest.mock import patch

GRACEFUL_FAILURE_MSG = (
    "I encountered an unexpected error. Your query has been logged. "
    "Please try again or contact support@taskflowpro.com."
)

def safe_run_agent_turn(executor, memory, user_input: str) -> str:
    """Wrapper that catches unexpected errors and returns a graceful message."""
    try:
        return run_agent_turn(executor, memory, user_input)
    except Exception as exc:
        print(f"[ERROR] Agent raised: {type(exc).__name__}: {exc}")
        return GRACEFUL_FAILURE_MSG


# Test 1: Normal operation
mem = AgentMemory()
print("Normal operation:")
resp = safe_run_agent_turn(executor, mem, "How do I log time on a task?")
print(f"  {resp[:150]}")

# Test 2: Simulated LLM timeout / exception
print("\nSimulated LLM failure:")
with patch.object(executor, "invoke", side_effect=TimeoutError("LLM request timed out")):
    resp = safe_run_agent_turn(executor, mem, "How do I add a task?")
print(f"  {resp}")

Normal operation:
  To log time on a task in TaskFlow Pro, you can use the "Log Time" button available directly on the task. This feature is available on the Pro plan and

Simulated LLM failure:
[ERROR] Agent raised: TimeoutError: LLM request timed out
  I encountered an unexpected error. Your query has been logged. Please try again or contact support@taskflowpro.com.


## 8.5 Deployment Assumptions & Limitations

| Assumption | Detail |
|---|---|
| Authentication | Users are pre-authenticated; the API does not handle login |
| Session storage | Sessions are in-memory; a server restart clears all sessions |
| Vector store | ChromaDB is persisted to disk; no re-indexing needed between restarts |
| LLM provider | OpenAI API — requires valid key and internet access |
| Rate limits | OpenAI rate limits apply; not yet handled with retries (v1 limitation) |
| Scalability | Single-process; for production, use a distributed session store (Redis) |
| PII | Input/output PII is masked before logging but not encrypted at rest |

## 8.6 Running the API

```bash
# Install dependencies
pip install -r requirements.txt

# Set up environment
cp .env.example .env
# Edit .env and add your OPENAI_API_KEY

# Start the server
uvicorn main:app --reload --port 8000

# Test
curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "How do I add a sub-task?"}'
```